# Particle filter test (simulated)

Simulates a robot driving straight â†’ turning â†’ straight. The camera is **hidden** for a few seconds in the middle.

- The **true robot** is slightly different from the model (faster, turns more), like in the odometry tests.
- The **camera** sees the true pose with a bit of noise, or nothing (`None`) while hidden.
- The filter only gets the wheel commands and the camera.

Expected: the particle cloud **spreads** while the camera is hidden, and **shrinks** back when it returns.

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
from loguru import logger

sys.path.insert(0, "../../src")
from data_types import SensorData, State  # noqa: E402
from pose_estimation import PoseEstimator  # noqa: E402

logger.remove()

In [ ]:
# Scenario
DT = 0.05
PLAN = [(100, 100, 6.0), (75, -75, 3.0), (100, 100, 12.0)]  # (left, right, seconds)
HIDDEN = (4.0, 16.0)  # camera hidden between these times (s)

TRUE_V_FACTOR = 1.09  # real robot vs model (odometry tests: ~9% faster, ~8% more turning)
TRUE_W_FACTOR = 1.08
CAM_POS_STD, CAM_HEADING_STD = 0.5, 1.0  # simulated camera noise (cm, deg)
SNAPSHOTS = {"camera hidden": HIDDEN[0], "end of outage": HIDDEN[1] - DT, "camera back (+1 s)": HIDDEN[1] + 1.0}

In [ ]:
rng = np.random.default_rng(1)
pe = PoseEstimator(seed=0)
cal = pe._config["calibration"]

true = np.zeros(3)  # true pose [x, y, heading]
log = {"t": [], "true": [], "est": [], "std": [], "cam": []}
snapshots = {}

t = 0.0
for left, right, duration in PLAN:
    for _ in range(round(duration / DT)):
        # True robot motion (same equations, different coefficients)
        v = 0.5 * (left + right) * cal["v_coef"] * TRUE_V_FACTOR
        w = (left - right) * cal["w_coef"] * TRUE_W_FACTOR
        mid = np.radians(true[2] + 0.5 * w * DT)
        true += [v * DT * np.sin(mid), -v * DT * np.cos(mid), w * DT]

        # Camera: noisy true pose, or None while hidden
        hidden = HIDDEN[0] <= t < HIDDEN[1]
        cam = (
            None
            if hidden
            else State(
                position=tuple(true[:2] + rng.normal(0, CAM_POS_STD, 2)),
                heading=float(true[2] + rng.normal(0, CAM_HEADING_STD)),
            )
        )

        pe.move(left, right, DT)
        pe.sense(SensorData([0] * 7, [0, 0], [left, right], DT, cam))

        est = pe.get_estimate()
        log["t"].append(t)
        log["true"].append(true.copy())
        log["est"].append([*est.position, est.heading])
        log["std"].append(pe.particles[:, :2].std(axis=0).mean())
        log["cam"].append(None if cam is None else cam.position)
        for name, when in SNAPSHOTS.items():
            if abs(t - when) < DT / 2:
                snapshots[name] = pe.particles
        t += DT

T, TRUE, EST, STD = (np.array(log[k]) for k in ("t", "true", "est", "std"))
err = np.hypot(*(EST[:, :2] - TRUE[:, :2]).T)
print(f"Max position error while hidden: {err[(T >= HIDDEN[0]) & (T < HIDDEN[1])].max():.1f} cm")
print(f"Position error at the end:       {err[-1]:.1f} cm")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: paths and particle clouds
ax = axes[0]
for (name, p), color in zip(snapshots.items(), ["tab:orange", "tab:red", "tab:green"], strict=False):
    ax.scatter(p[:, 0], p[:, 1], s=2, alpha=0.3, color=color, label=f"particles: {name}")
cams = np.array([c for c in log["cam"] if c is not None])
ax.scatter(cams[:, 0], cams[:, 1], s=4, color="gray", alpha=0.4, label="camera")
ax.plot(TRUE[:, 0], TRUE[:, 1], "k-", label="true path")
ax.plot(EST[:, 0], EST[:, 1], "b--", label="estimate")
ax.invert_yaxis()
ax.set_aspect("equal")
ax.set_xlabel("x (cm)")
ax.set_ylabel("y (cm)")
ax.legend(fontsize=8)
ax.set_title("Paths and particle clouds")

# Right: error and cloud size over time
ax = axes[1]
ax.axvspan(*HIDDEN, color="gray", alpha=0.2, label="camera hidden")
ax.plot(T, err, label="position error (cm)")
ax.plot(T, STD, label="particle spread (cm)")
ax.set_xlabel("time (s)")
ax.legend()
ax.set_title("Error and uncertainty")
plt.tight_layout()
plt.show()